# Reading compliance: two bounded experiments
The earlier GPU access limit has cleared. No new loss or quality results exist merely from preparing this notebook. This notebook executes only the approved fresh3-epoch LoRA runs (300 and600 examples), preserves all epoch checkpoints, and generates the current-adapter comparator plus six checkpoints on the same34 frozen inputs. No paid calls, purchases, test access, or UI adapter changes.

Select the existing free T4 runtime. The same FP16 precision and allocator setting apply to both runs. Stop on any failed gate. The existing completed historical adapter is used only as the evaluation comparator; both training runs start fresh. All data is independently agent-reviewed, with human review pending.


In [ ]:
import os, sys
assert 'torch' not in sys.modules, 'Restart session; allocator must precede torch import'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
from pathlib import Path
import hashlib, json, zipfile, subprocess
EXPECTED_KIT_SHA256='24be38ca94a924e635e6f06c7086cef1db1b41f0803e8d3ec411482d230424fb'
KIT=Path('/content/reading-compliance-20261031-002')


In [ ]:
# Upload reading-compliance-colab.zip, or use Colab Files sidebar before running this cell.
ARCHIVE=Path('/content/reading-compliance-colab.zip')
if not ARCHIVE.exists():
    from google.colab import files
    uploaded=files.upload()
    assert set(uploaded)=={'reading-compliance-colab.zip'}
    ARCHIVE.write_bytes(uploaded['reading-compliance-colab.zip'])
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()==EXPECTED_KIT_SHA256
assert not KIT.exists(), 'Preserve existing kit; restart runtime for a fresh extraction'
with zipfile.ZipFile(ARCHIVE) as archive:
    assert len(archive.namelist())==len(set(archive.namelist()))
    assert archive.testzip() is None
    for info in archive.infolist():
        assert not Path(info.filename).is_absolute() and '..' not in Path(info.filename).parts
        assert (info.external_attr>>16)&0o170000 != 0o120000
    archive.extractall(KIT)
kit_manifest=json.loads((KIT/'kit-manifest.json').read_text())
assert set(archive.namelist())==set(kit_manifest['files'])|{'kit-manifest.json'}
for name,expected in kit_manifest['files'].items():
    assert hashlib.sha256((KIT/name).read_bytes()).hexdigest()==expected,name
STAGE=KIT/'stage1'
EXPERIMENT=STAGE/'experiments'/'reading-compliance-20261031-002'
assert hashlib.sha256((EXPERIMENT/'plan.json').read_bytes()).hexdigest()==kit_manifest['plan_sha256']


In [ ]:
# Official registry pins; torch>=2.6 is required for safe optimizer/RNG resume.
# Reuse the hash-bound original dependency preflight ONLY, preserving its torchao safeguards.
TRAINING_MODE='lora'
legacy_notebook=json.loads((STAGE/'sft_colab.ipynb').read_text())
exec(compile(''.join(legacy_notebook['cells'][4]['source']),'reviewed-dependency-preflight','exec'))
subprocess.run([sys.executable,'-m','pip','install','-q','matplotlib'],check=True)
from packaging.version import Version
import torch
assert Version(torch.__version__.split('+')[0])>=Version('2.6')
assert torch.cuda.is_available() and torch.cuda.device_count()==1
assert 'T4' in torch.cuda.get_device_name(0), 'No automatic hardware substitution'
print(torch.__version__,torch.cuda.get_device_name(0))


In [ ]:
# Prefetch pinned tokenizer; existing CPU mask tests deliberately require local_files_only.
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B',revision='70d244cc86ccca08cf5af4e1e306ecf908b1ad5e')
# Offline integrity tests include real tiny CPU models; never load held-out files.
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(STAGE),'-p','test_sft_train.py','-v'],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(EXPERIMENT),'-p','test_*.py','-v'],check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT=Path('/content/drive/MyDrive/Reading-Model/sft/reading-compliance-20261031-002')
CURRENT_ADAPTER=Path('/content/drive/MyDrive/Reading-Model/sft/qwen-sft300-v2-lora-allocator-20261030-008/full/adapter')
assert CURRENT_ADAPTER.is_dir(), 'Locate the historical verified adapter; never substitute another adapter'
OUTPUT.mkdir(parents=True,exist_ok=True)
if not (OUTPUT/'dependency-preflight.json').exists():
    (OUTPUT/'dependency-preflight.json').write_text(json.dumps(PIP_CHECK_DIAGNOSTICS,indent=2)+'\n')
# Explicit resume only: set BOTH only and exact latest complete epoch checkpoint.
ONLY=None # 'same300' or 'expanded600'
RESUME=None # exact OUTPUT/ONLY/full/checkpoints/checkpoint-N


In [ ]:
command=[sys.executable,str(EXPERIMENT/'execute.py'),'--plan',str(EXPERIMENT/'plan.json'),'--output',str(OUTPUT),'--current-adapter',str(CURRENT_ADAPTER)]
if ONLY is not None: command.extend(['--only',ONLY])
if RESUME is not None:
    assert ONLY is not None
    command.extend(['--resume',str(RESUME)])
# Fresh smoke per configuration; >=15% operational headroom; zero skipped updates.
# Stops preserve numerical/memory/integrity diagnostics. Never continue from old completed adapter.
subprocess.run(command,check=True)


In [ ]:
# Plot only real recorded update losses. Prior plots are never replaced.
for name in ('same300','expanded600'):
    full=OUTPUT/name/'full'
    if (full/'run-record.json').exists() and json.loads((full/'run-record.json').read_text())['status']=='training_completed':
        plot=OUTPUT/name/'loss-curves'
        if not plot.exists():
            subprocess.run([sys.executable,str(EXPERIMENT/'plot_results.py'),'loss','--full',str(full),'--out',str(plot)],check=True)


In [ ]:
# Conceal identity-key.json and all run paths from the independent reviewing agent.
# Supply only blind-review/review.jsonl + full prompt/rubric + REVIEW_PROTOCOL.md.
plan=json.loads((EXPERIMENT/'plan.json').read_text())
subprocess.run([sys.executable,str(EXPERIMENT/'evaluate.py'),'blind','--reference',str(EXPERIMENT/'frozen'),'--reference-sha',plan['frozen_manifest_sha256'],'--runs',str(OUTPUT/'review-runs.json'),'--out',str(OUTPUT/'blind-review')],check=True)
print('Raw generation complete. Independent blinded quality ratings and human review remain pending.')


After independent ratings, use `evaluate.py summarize`, `plot_results.py pairs`, and `choose_checkpoint.py` as documented in README.md. A qualifying >=+20pp primary gain needs two more paired seeds on the same inputs, positive gain in both, and no new serious failures. Extra seeds are not independent cases. No automatic adapter replacement.

The fresh epoch-one checkpoint uses linear decay over114updates, unlike the historical38update schedule. The600row experiment has225updates and1800row exposures vs114updates/900exposures; it changes coverage and optimization exposure together.
